# 🛍️ Fashion Commerce: Customer Churn & LTV Machine Learning Model

## Business Context & Problem Statement
Customer acquisition costs (CAC) in fashion e-commerce have surged over recent years. Retaining existing active customers and proactively identifying accounts at risk of becoming inactive is vital for marketing ROI.

- **Dataset**: Integrated warehouse tables (`gold.dim_customers` and `gold.fct_orders`) in `data/warehouse.duckdb`.
- **Target Definition (Question C1)**: Non-contractual churn defined by a **180-day inactivity threshold** (Lapsed = 1, Active/At Risk = 0) measured relative to the project reference date.
- **Algorithm**: **LightGBM Classifier Pipeline** with categorical encoding, numerical scaling, and stratified cross-validation.

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    roc_curve,
)
from lightgbm import LGBMClassifier

# Visualization settings
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Load Clean Feature Set from DuckDB Warehouse
We extract customer attributes from `gold.dim_customers` joined with behavioral aggregations from `gold.fct_orders`.

In [ ]:
DB_PATH = Path("../data/warehouse.duckdb")
con = duckdb.connect(str(DB_PATH), read_only=True)

query = """
WITH cust_orders AS (
    SELECT
        customer_id,
        count(distinct booking_id) as total_orders,
        round(avg(line_total), 2) as avg_line_total,
        round(avg(delivery_distance_km), 1) as avg_distance_km,
        round(avg(shipment_fee), 2) as avg_shipment_fee,
        round(100.0 * count(distinct case when promo_code != 'NO_PROMO' then booking_id end) / nullif(count(distinct booking_id), 0), 1) as promo_orders_pct,
        mode(payment_method) as preferred_payment_method
    FROM gold.fct_orders
    WHERE payment_status = 'Success'
    GROUP BY customer_id
)
SELECT
    c.customer_id,
    c.gender,
    c.device_type,
    c.home_location,
    c.age,
    c.tenure_days,
    c.total_orders,
    c.total_units_purchased,
    c.lifetime_spend,
    c.avg_order_value,
    c.is_repeat_buyer,
    c.f_score,
    c.m_score,
    coalesce(o.avg_distance_km, 50.0) as avg_distance_km,
    coalesce(o.avg_shipment_fee, 0.0) as avg_shipment_fee,
    coalesce(o.promo_orders_pct, 0.0) as promo_orders_pct,
    coalesce(o.preferred_payment_method, 'Credit Card') as preferred_payment_method,
    -- Churn definition (180-day inactivity): Lapsed = 1, Active/At Risk = 0
    CASE WHEN c.customer_activity_status = 'Lapsed' THEN 1 ELSE 0 END as is_churned
FROM gold.dim_customers c
LEFT JOIN cust_orders o ON c.customer_id = o.customer_id
WHERE c.total_orders > 0
"""

df = con.execute(query).df()
con.close()
print(f"Extracted {len(df):,} customer records with {df.shape[1]} features.")
df.head()

## 2. Exploratory Data Analysis: Churn Patterns
Inspect class balance and compare behavioral patterns across churned vs active cohorts.

In [ ]:
# Class Distribution
churn_counts = df["is_churned"].value_counts(normalize=True) * 100
print(f"Active/At Risk (0): {churn_counts[0]:.1f}%")
print(f"Churned/Lapsed  (1): {churn_counts[1]:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total Orders by Churn Status
sns.boxplot(data=df, x="is_churned", y="total_orders", showfliers=False, ax=axes[0], palette=["#3B82F6", "#EF4444"])
axes[0].set_title("Order Frequency by Churn Status")
axes[0].set_xticklabels(["Retained (0)", "Churned (1)"])

# Lifetime Spend by Churn Status
sns.boxplot(data=df, x="is_churned", y="lifetime_spend", showfliers=False, ax=axes[1], palette=["#3B82F6", "#EF4444"])
axes[1].set_title("Lifetime Spend by Churn Status")
axes[1].set_xticklabels(["Retained (0)", "Churned (1)"])

plt.tight_layout()
plt.show()

## 3. Feature Selection & Train/Test Split
Notice that we purposefully **exclude direct recency days** (`days_since_last_order` and `r_score`) from the feature matrix to eliminate target leakage and ensure the model learns organic customer persona and engagement drivers.

In [ ]:
categorical_cols = ["gender", "device_type", "home_location", "preferred_payment_method"]
numerical_cols = [
    "age",
    "tenure_days",
    "total_orders",
    "total_units_purchased",
    "lifetime_spend",
    "avg_order_value",
    "is_repeat_buyer",
    "f_score",
    "m_score",
    "avg_distance_km",
    "avg_shipment_fee",
    "promo_orders_pct",
]
feature_cols = categorical_cols + numerical_cols

X = df[feature_cols]
y = df["is_churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training set: {X_train.shape[0]:,} samples | Testing set: {X_test.shape[0]:,} samples")

## 4. Pipeline Construction & Model Training
We build an end-to-end `scikit-learn` Pipeline with `StandardScaler` for numerical fields, `OneHotEncoder` for categorical fields, and a tuned `LightGBMClassifier`.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
    ]
)

clf = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=-1,
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", clf),
    ]
)

pipeline.fit(X_train, y_train)
print("LightGBM Pipeline trained successfully!")

## 5. Model Evaluation: ROC-AUC, Precision, Recall & F1

In [ ]:
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_prob)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"ROC-AUC Score: {auc:.4f}")
print(f"Accuracy Score: {acc:.4f}")
print(f"F1 Score:       {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Retained", "Churned"]))

# Plot ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color="#3B82F6", lw=2, label=f"LightGBM ROC (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], color="gray", linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Receiver Operating Characteristic (ROC) Curve")
plt.legend(loc="lower right")
plt.show()

## 6. Feature Importance & Interpretability
Identify which customer behaviors contribute most to churn risk predictions.

In [ ]:
encoder = pipeline.named_steps["preprocessor"].named_transformers_["cat"]
cat_names = list(encoder.get_feature_names_out(categorical_cols))
all_names = numerical_cols + cat_names
importances = pipeline.named_steps["classifier"].feature_importances_

feat_df = pd.DataFrame({"Feature": all_names, "Importance": importances})
feat_df = feat_df.sort_values("Importance", ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_df, x="Importance", y="Feature", palette="Blues_r")
plt.title("Top 15 Feature Importances in LightGBM Churn Classifier")
plt.show()

## 7. Model Explainability with SHAP (SHapley Additive exPlanations)
Explain global feature importance and individual sample predictions using game-theoretic Shapley values via `shap.TreeExplainer`.

In [ ]:
import shap

clf_step = pipeline.named_steps['classifier']
prep_step = pipeline.named_steps['preprocessor']

explainer = shap.TreeExplainer(clf_step)
X_sample = X_test.sample(n=min(500, len(X_test)), random_state=42)
X_sample_trans = prep_step.transform(X_sample)
shap_values = explainer.shap_values(X_sample_trans)

print('Base expected log-odds value:', explainer.expected_value)
mean_shap = np.mean(np.abs(shap_values), axis=0)
shap_rank = pd.DataFrame({'Feature': all_names, 'Mean_Abs_SHAP': mean_shap}).sort_values('Mean_Abs_SHAP', ascending=False)
print('Top 10 Global SHAP Drivers:')
print(shap_rank.head(10).to_string(index=False))


## 8. Customer Lifetime Value (LTV) Regressor (LightGBM)
Predict expected 12-month customer monetary yield using `LGBMRegressor` on log-transformed spend (`log1p(lifetime_spend)`).

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

ltv_num_cols = [c for c in numerical_cols if c not in ['lifetime_spend', 'm_score']]
X_ltv = df[categorical_cols + ltv_num_cols]
y_ltv = np.log1p(df['lifetime_spend'])

X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_ltv, y_ltv, test_size=0.2, random_state=42)

ltv_prep = ColumnTransformer([
    ('num', StandardScaler(), ltv_num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
])
ltv_pipe = Pipeline([('preprocessor', ltv_prep), ('regressor', LGBMRegressor(n_estimators=250, learning_rate=0.05, max_depth=6, random_state=42, verbosity=-1))])
ltv_pipe.fit(X_tr_l, y_tr_l)

y_pred_log = ltv_pipe.predict(X_te_l)
y_pred_idr = np.expm1(y_pred_log)
y_true_idr = np.expm1(y_te_l)

print('LTV R2 Score:', r2_score(y_te_l, y_pred_log))
print('LTV MAE (IDR):', mean_absolute_error(y_true_idr, y_pred_idr))
print('LTV RMSE (IDR):', root_mean_squared_error(y_true_idr, y_pred_idr))


## 9. Model Serialization & Export for Streamlit App
Persist both pipelines and evaluation metadata into `app/models/` for live scoring, interactive SHAP waterfall inspection, and counterfactual simulation.

In [ ]:
MODEL_DIR = Path('../app/models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(pipeline, MODEL_DIR / 'churn_lgbm_model.joblib')
joblib.dump(ltv_pipe, MODEL_DIR / 'ltv_lgbm_model.joblib')
print('Dual pipelines saved to', MODEL_DIR.resolve())
